In [1]:
!pip install -q pypdf langchain-text-splitters sentence-transformers google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 3.6 MB/s eta 0:00:00


In [1]:
import os
import re
import urllib.request
import numpy as np

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from google.colab import userdata
from google import genai


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

URL_PDF = (
    "https://raw.githubusercontent.com/"
    "kimberlyn05/sales-knowledge-ai-agent/"
    "main/data/documents/Politicas_Internas_NexaShip.pdf"
)

RUTA_PDF = "/content/Politicas_Internas_NexaShip.pdf"

MODELO_EMBEDDINGS = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)


# ============================================================
# 2. DESCARGAR PDF
# ============================================================

if not os.path.exists(RUTA_PDF):
    urllib.request.urlretrieve(URL_PDF, RUTA_PDF)
    print("✓ PDF descargado desde GitHub")
else:
    print("✓ PDF disponible")


# ============================================================
# 3. EXTRAER TEXTO POR PÁGINA
# ============================================================

lector = PdfReader(RUTA_PDF)

paginas = []

for numero_pagina, pagina in enumerate(lector.pages, start=1):
    texto = pagina.extract_text() or ""

    paginas.append({
        "pagina": numero_pagina,
        "texto": texto
    })

print(f"✓ Páginas procesadas: {len(paginas)}")


# ============================================================
# 4. LIMPIAR TEXTO
# ============================================================

def limpiar_texto(texto):
    texto = texto.replace("\x7f", " ")
    texto = texto.replace("\u00a0", " ")

    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)

    return texto.strip()


paginas_limpias = []

for pagina in paginas:
    paginas_limpias.append({
        "pagina": pagina["pagina"],
        "texto": limpiar_texto(pagina["texto"])
    })

print("✓ Texto limpiado")


# ============================================================
# 5. CREAR CHUNKS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks_limpios = []
chunk_id = 1

for pagina in paginas_limpias:
    fragmentos = text_splitter.split_text(
        pagina["texto"]
    )

    for fragmento in fragmentos:
        chunks_limpios.append({
            "chunk_id": chunk_id,
            "pagina": pagina["pagina"],
            "texto": fragmento
        })

        chunk_id += 1

print(f"✓ Chunks creados: {len(chunks_limpios)}")


# ============================================================
# 6. CARGAR MODELO DE EMBEDDINGS
# ============================================================

modelo_embeddings = SentenceTransformer(
    MODELO_EMBEDDINGS
)

textos_chunks = [
    chunk["texto"]
    for chunk in chunks_limpios
]

embeddings_chunks = modelo_embeddings.encode(
    textos_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    f"✓ Embeddings generados: "
    f"{len(embeddings_chunks)}"
)


# ============================================================
# 7. CONFIGURAR GEMINI
# ============================================================

gemini_api_key = userdata.get(
    "GEMINI_API_KEY"
)

cliente_gemini = genai.Client(
    api_key=gemini_api_key
)

print("✓ Cliente Gemini configurado")


# ============================================================
# 8. FUNCIÓN DE BÚSQUEDA SEMÁNTICA
# ============================================================

def buscar_evidencia(pregunta, top_k=5):

    embedding_pregunta = modelo_embeddings.encode(
        [pregunta],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    similitudes = (
        embeddings_chunks
        @ embedding_pregunta
    )

    indices_relevantes = np.argsort(
        similitudes
    )[::-1][:top_k]

    resultados = []

    for indice in indices_relevantes:
        chunk = chunks_limpios[indice]

        resultados.append({
            "chunk_id": chunk["chunk_id"],
            "pagina": chunk["pagina"],
            "similitud": float(
                similitudes[indice]
            ),
            "texto": chunk["texto"]
        })

    return resultados


print("✓ Búsqueda semántica preparada")


# ============================================================
# 9. FUNCIÓN DEL AGENTE
# ============================================================

def responder_con_evidencia(
    pregunta,
    top_k=5
):

    evidencias = buscar_evidencia(
        pregunta,
        top_k=top_k
    )

    contexto = "\n\n".join(
        [
            (
                f"[Fuente: página "
                f"{evidencia['pagina']}]\n"
                f"{evidencia['texto']}"
            )
            for evidencia in evidencias
        ]
    )

    prompt = f"""
Eres Sales Knowledge AI Agent,
un asistente interno de NexaShip.

Tu tarea es responder preguntas
utilizando exclusivamente la evidencia
documental proporcionada.

Reglas obligatorias:
- No inventes información.
- No utilices conocimiento externo.
- No completes datos faltantes mediante suposiciones.
- Si la evidencia no permite responder con seguridad,
  indícalo claramente.
- Responde en español.
- Sé claro y profesional.
- Menciona las páginas utilizadas como fuente.
- Si existen varias reglas relevantes,
  intégralas en una sola respuesta coherente.

PREGUNTA:
{pregunta}

EVIDENCIA DOCUMENTAL:
{contexto}

RESPUESTA:
"""

    respuesta = (
        cliente_gemini.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
    )

    return {
        "pregunta": pregunta,
        "respuesta": respuesta.text,
        "evidencias": evidencias
    }


print("✓ Agente configurado")
print()
print("🚀 Sales Knowledge AI Agent listo")

✓ PDF descargado desde GitHub
✓ Páginas procesadas: 10
✓ Texto limpiado
✓ Chunks creados: 33


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Embeddings generados: 33
✓ Cliente Gemini configurado
✓ Búsqueda semántica preparada
✓ Agente configurado

🚀 Sales Knowledge AI Agent listo


In [2]:
def clasificar_evidencia(evidencias):
    mejor_score = evidencias[0]["similitud"]

    if mejor_score >= 0.45:
        estado = "fuerte"
    elif mejor_score >= 0.30:
        estado = "incierta"
    else:
        estado = "insuficiente"

    return {
        "estado": estado,
        "mejor_score": mejor_score
    }


print("✓ Clasificador de evidencia configurado")

✓ Clasificador de evidencia configurado


In [3]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.2 MB/s eta 0:00:00


In [4]:
from google.colab import userdata
from groq import Groq

# Obtener la API key desde los Secrets de Colab
groq_api_key = userdata.get("GROQ_API_KEY")

# Crear cliente de Groq
cliente_groq = Groq(
    api_key=groq_api_key
)

print("✓ Cliente Groq configurado correctamente")

✓ Cliente Groq configurado correctamente


In [5]:
def generar_respuesta_llm(prompt):
    """
    Intenta generar la respuesta con Gemini.
    Si Gemini falla, utiliza Groq como fallback.
    """

    try:
        # Proveedor principal: Gemini
        respuesta_gemini = cliente_gemini.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        return {
            "texto": respuesta_gemini.text,
            "proveedor": "gemini",
            "fallback_usado": False
        }

    except Exception as error_gemini:
        print(
            "⚠ Gemini no disponible. "
            "Intentando con Groq..."
        )

        try:
            # Proveedor alternativo: Groq
            respuesta_groq = cliente_groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0
            )

            return {
                "texto": respuesta_groq.choices[0].message.content,
                "proveedor": "groq",
                "fallback_usado": True
            }

        except Exception as error_groq:
            return {
                "texto": (
                    "No fue posible generar una respuesta "
                    "en este momento porque los proveedores "
                    "de lenguaje no están disponibles."
                ),
                "proveedor": None,
                "fallback_usado": True,
                "error_gemini": str(error_gemini),
                "error_groq": str(error_groq)
            }


print("✓ Fallback Gemini → Groq configurado")

✓ Fallback Gemini → Groq configurado


In [6]:
def responder_agente_controlado(
    pregunta,
    top_k=5
):
    # 1. Recuperar evidencia
    evidencias = buscar_evidencia(
        pregunta,
        top_k=top_k
    )

    # 2. Clasificar evidencia
    clasificacion = clasificar_evidencia(
        evidencias
    )

    estado = clasificacion["estado"]
    mejor_score = clasificacion["mejor_score"]

    # 3. Bloquear consultas claramente sin evidencia
    # Aquí NO se llama a ningún LLM
    if estado == "insuficiente":
        return {
            "pregunta": pregunta,
            "respuesta": (
                "No encontré evidencia documental suficiente "
                "en las fuentes disponibles para responder "
                "esta consulta con seguridad."
            ),
            "estado_evidencia": estado,
            "mejor_score": mejor_score,
            "evidencias": evidencias,
            "llamada_llm": False,
            "proveedor": None,
            "fallback_usado": False
        }

    # 4. Construir contexto documental
    contexto = "\n\n".join(
        [
            (
                f"[Fuente: página "
                f"{evidencia['pagina']}]\n"
                f"{evidencia['texto']}"
            )
            for evidencia in evidencias
        ]
    )

    # 5. Instrucciones según nivel de evidencia
    if estado == "incierta":
        instruccion_evidencia = """
La relevancia de la evidencia recuperada es incierta.

Debes ser especialmente conservador:
- Responde solo si la evidencia contiene información
  directamente aplicable a la pregunta.
- No deduzcas reglas que no estén expresadas.
- No conviertas similitudes temáticas en hechos.
- Si la evidencia no responde directamente,
  indica que no existe información suficiente.
"""
    else:
        instruccion_evidencia = """
La evidencia recuperada presenta una relevancia fuerte.

Aun así:
- Responde exclusivamente con base en la evidencia.
- No agregues información externa.
- No realices suposiciones no respaldadas.
"""

    # 6. Construir prompt
    prompt = f"""
Eres Sales Knowledge AI Agent,
un asistente interno de NexaShip.

Tu tarea es responder preguntas utilizando
exclusivamente la evidencia documental proporcionada.

Reglas obligatorias:
- No inventes información.
- No utilices conocimiento externo.
- No completes datos faltantes mediante suposiciones.
- Si la evidencia no permite responder con seguridad,
  indícalo claramente.
- Responde en español.
- Sé claro y profesional.
- Menciona las páginas utilizadas como fuente.

ESTADO DE EVIDENCIA:
{estado}

INSTRUCCIONES ADICIONALES:
{instruccion_evidencia}

PREGUNTA:
{pregunta}

EVIDENCIA DOCUMENTAL:
{contexto}

RESPUESTA:
"""

    # 7. Generar respuesta con fallback automático
    resultado_llm = generar_respuesta_llm(
        prompt
    )

    # 8. Retornar resultado completo
    return {
        "pregunta": pregunta,
        "respuesta": resultado_llm["texto"],
        "estado_evidencia": estado,
        "mejor_score": mejor_score,
        "evidencias": evidencias,
        "llamada_llm": True,
        "proveedor": resultado_llm["proveedor"],
        "fallback_usado": resultado_llm["fallback_usado"]
    }


print("✓ Agente controlado con fallback configurado")

✓ Agente controlado con fallback configurado


In [7]:
resultado_final = responder_agente_controlado(
    "¿Puedo prometerle a un cliente una integración personalizada?"
)

print("PREGUNTA:")
print(resultado_final["pregunta"])

print("\nRESPUESTA:")
print(resultado_final["respuesta"])

print("\nESTADO DE EVIDENCIA:")
print(resultado_final["estado_evidencia"])

print("\nMEJOR SCORE:")
print(f"{resultado_final['mejor_score']:.4f}")

print("\n¿SE LLAMÓ AL LLM?")
print(resultado_final["llamada_llm"])

print("\nPROVEEDOR:")
print(resultado_final["proveedor"])

print("\n¿SE USÓ FALLBACK?")
print(resultado_final["fallback_usado"])

PREGUNTA:
¿Puedo prometerle a un cliente una integración personalizada?

RESPUESTA:
No, no puedes prometerle a un cliente una integración personalizada.

Según la política de NexaShip:
*   El personal comercial no podrá comprometer, sin autorización correspondiente, las fechas de desarrollo de integraciones personalizadas (Página 4).
*   Cuando un prospecto utilice una arquitectura personalizada o requerimientos técnicos fuera de las capacidades documentadas, la persona comercial deberá registrar el requerimiento y solicitar validación especializada antes de confirmar viabilidad, alcance o fechas (Página 4).
*   Toda respuesta proporcionada a un prospecto o cliente debe estar sustentada en información oficialmente documentada, validada o aprobada por el área responsable (Página 3).
*   Las personas del equipo comercial son responsables de evitar compromisos no autorizados (Página 9).

ESTADO DE EVIDENCIA:
fuerte

MEJOR SCORE:
0.6006

¿SE LLAMÓ AL LLM?
True

PROVEEDOR:
gemini

¿SE USÓ F